In [ ]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [ ]:
import sys
import importlib

# Add the Silver layer directory to sys.path BEFORE importing
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
import utils_silver
from utils_silver import *

importlib.reload(utils_silver)

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/02_Silver_Layer/utils_silver.py


In [8]:
DB_GOLD = "bupa_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DB_GOLD}.fact_members
    USING DELTA
    LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_members'
""")

spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {DB_GOLD}.dim_member_segment
        USING DELTA
        LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/dim_member_segment'
    """)

spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {DB_GOLD}.dim_region
        USING DELTA
        LOCATION 'abfss://golddata@clientdatastorage.dfs.core.windows.net/dim_region'
    """)

DataFrame[]

# Cell 1 — Imports, config, load fact & dimensions

In [9]:
# Cell 1 — Setup & load sources for star_members

from pyspark.sql import functions as F

DB_GOLD   = "bupa_gold"
GOLD_BASE = "abfss://golddata@clientdatastorage.dfs.core.windows.net"

print("DB_GOLD   :", DB_GOLD)
print("GOLD_BASE :", GOLD_BASE)

# --- Load fact & dimensions from Gold ---

fact_members = spark.table(f"{DB_GOLD}.fact_members")
dim_segment  = spark.table(f"{DB_GOLD}.dim_member_segment")
dim_region   = spark.table(f"{DB_GOLD}.dim_region")

print("fact_members rowcount:", fact_members.count())
fact_members.printSchema()

print("\nDim member segment:")
dim_segment.show(5, truncate=False)
dim_segment.printSchema()

print("\nDim region:")
dim_region.show(5, truncate=False)
dim_region.printSchema()


DB_GOLD   : bupa_gold
GOLD_BASE : abfss://golddata@clientdatastorage.dfs.core.windows.net
fact_members rowcount: 381109
root
 |-- Member_ID: string (nullable = true)
 |-- Policy_ID: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker_Flag: string (nullable = true)
 |-- Chronic_Disease: string (nullable = true)
 |-- Employment_Status: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Age_Band: string (nullable = true)
 |-- BMI_Band: string (nullable = true)
 |-- Smoker_Status: string (nullable = true)
 |-- Chronic_Flag: integer (nullable = true)
 |-- Chronic_Group: string (nullable = true)
 |-- Employment_Group: string (nullable = true)
 |-- Region_Group: string (nullable = true)
 |-- dq_age_valid: integer (nullable = true)
 |-- dq_bmi_valid: integer (nullable = true)


Dim member segment:


+------------------+--------+----------+-------------+------------+-------------+----------------+--------------+
|Member_Segment_Key|Age_Band|BMI_Band  |Smoker_Status|Chronic_Flag|Chronic_Group|Employment_Group|Region_Group  |
+------------------+--------+----------+-------------+------------+-------------+----------------+--------------+
|0                 |18–29   |Obese     |Non-Smoker   |1           |Hypertension |Employed        |Region_150-249|
|1                 |18–29   |Overweight|Smoker       |0           |None         |Employed        |Region_350+   |
|2                 |45–59   |Obese     |Non-Smoker   |1           |Diabetes     |Retired         |Region_0-49   |
|3                 |18–29   |Obese     |Smoker       |1           |Asthma       |Retired         |Region_50-149 |
|4                 |45–59   |Overweight|Smoker       |1           |Hypertension |Student         |Region_50-149 |
+------------------+--------+----------+-------------+------------+-------------+-------

+------+-----------+------------+
|Region|Region_Code|Region_Label|
+------+-----------+------------+
|00    |00         |Region_00   |
|10    |10         |Region_10   |
|100   |100        |Region_100  |
|110   |110        |Region_110  |
|120   |120        |Region_120  |
+------+-----------+------------+
only showing top 5 rows

root
 |-- Region: string (nullable = true)
 |-- Region_Code: string (nullable = true)
 |-- Region_Label: string (nullable = true)



# Cell 2 — Build star_members by joining dims

This cell is a bit defensive: it auto-detects join columns based on what’s actually in your tables.

In [10]:
# Cell 2 — Build star_members (one row per member, enriched with dims)

fm = fact_members.alias("fm")
ds = dim_segment.alias("ds")
dr = dim_region.alias("dr")

# --- Detect join keys for member segment ---
fm_cols = fm.columns
ds_cols = ds.columns
dr_cols = dr.columns

# segment key in fact
if "Member_Segment_Code" in fm_cols:
    fm_seg_col = "Member_Segment_Code"
elif "Segment_Code" in fm_cols:
    fm_seg_col = "Segment_Code"
elif "Member_Segment" in fm_cols:
    fm_seg_col = "Member_Segment"
else:
    fm_seg_col = None

# segment key in dim
if "Segment_Code" in ds_cols:
    ds_seg_col = "Segment_Code"
elif "Member_Segment" in ds_cols:
    ds_seg_col = "Member_Segment"
else:
    ds_seg_col = None

# region key in fact
if "Region_Code" in fm_cols:
    fm_region_col = "Region_Code"
else:
    fm_region_col = "Region" if "Region" in fm_cols else None

# region key in dim
if "Region_Code" in dr_cols:
    dr_region_col = "Region_Code"
else:
    dr_region_col = "Region" if "Region" in dr_cols else None

print("Using segment join:", fm_seg_col, "<->", ds_seg_col)
print("Using region  join:", fm_region_col, "<->", dr_region_col)

star_base = fm

# join segment dim if possible
if fm_seg_col and ds_seg_col:
    star_base = star_base.join(
        ds,
        F.col(f"fm.{fm_seg_col}") == F.col(f"ds.{ds_seg_col}"),
        how="left"
    )

# join region dim if possible
if fm_region_col and dr_region_col:
    star_base = star_base.join(
        dr,
        F.col(f"fm.{fm_region_col}") == F.col(f"dr.{dr_region_col}"),
        how="left"
    )

print("star_base rowcount:", star_base.count())

# --- Select & rename columns for star schema ---

select_cols = []

# grain
select_cols += [
    F.col("fm.Member_ID").alias("Member_ID"),
    F.col("fm.Policy_ID").alias("Policy_ID")
]

# demographic / risk features coming from fact
for c in [
    "Age", "Gender", "BMI", "Smoker",
    "Chronic_Disease", "Employment_Status"
]:
    if c in fm_cols:
        select_cols.append(F.col(f"fm.{c}"))

# region dimension columns (if available)
if "Region_Code" in dr_cols:
    select_cols.append(F.col("dr.Region_Code").alias("Region_Code"))
if "Region_Name" in dr_cols:
    select_cols.append(F.col("dr.Region_Name").alias("Region_Name"))
if "Region_Cluster" in dr_cols:
    select_cols.append(F.col("dr.Region_Cluster").alias("Region_Cluster"))

# member segment dimension columns (if available)
if "Segment_Code" in ds_cols:
    select_cols.append(F.col("ds.Segment_Code").alias("Member_Segment_Code"))
if "Segment_Name" in ds_cols:
    select_cols.append(F.col("ds.Segment_Name").alias("Member_Segment_Name"))
if "Segment_Description" in ds_cols:
    select_cols.append(F.col("ds.Segment_Description").alias("Member_Segment_Description"))

# any DQ flags present on fact_members
for dq_col in ["dq_age_valid", "dq_bmi_valid", "dq_region_valid"]:
    if dq_col in fm_cols:
        select_cols.append(F.col(f"fm.{dq_col}"))

# audit columns if present
for audit_col in ["created_at", "source_system", "record_hash"]:
    if audit_col in fm_cols:
        select_cols.append(F.col(f"fm.{audit_col}"))

star_members = star_base.select(*select_cols)

star_members.printSchema()
star_members.show(10, truncate=False)


Using segment join: None <-> None
Using region  join: Region <-> Region_Code


star_base rowcount: 381109
root
 |-- Member_ID: string (nullable = true)
 |-- Policy_ID: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Chronic_Disease: string (nullable = true)
 |-- Employment_Status: string (nullable = true)
 |-- Region_Code: string (nullable = true)
 |-- dq_age_valid: integer (nullable = true)
 |-- dq_bmi_valid: integer (nullable = true)



+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Chronic_Disease|Employment_Status|Region_Code|dq_age_valid|dq_bmi_valid|
+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Asthma         |Student          |280        |1           |1           |
|MEM_00001714|P_00001714|71 |M     |27.963719840043765|None           |Employed         |360        |1           |1           |
|MEM_00001746|P_00001746|75 |M     |30.541100957674118|Diabetes       |Student          |80         |1           |1           |
|MEM_00001967|P_00001967|37 |F     |25.982850362358867|Hypertension   |Employed         |280        |1           |1           |
|MEM_00001995|P_00001995|66 |M     |32.6294573451364  |Asthma         |Employed         |360        |1  

# Cell 3 — Write star_members to Gold container and register table + view

In [11]:
# Cell 3 — Persist star_members to Gold (Delta + view)

STAR_MEMBERS_PATH = f"{GOLD_BASE}/star_members"
print("Writing star_members to:", STAR_MEMBERS_PATH)

(
    star_members
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(STAR_MEMBERS_PATH)
)

print("✅ Wrote Delta files for star_members")

# Register as managed/external table
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")

spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.star_members")

spark.sql(f"""
CREATE TABLE {DB_GOLD}.star_members
USING DELTA
LOCATION '{STAR_MEMBERS_PATH}'
""")

print(f"✅ Registered table: {DB_GOLD}.star_members")

# Convenience view for BI tools / ad-hoc querying
spark.sql("DROP VIEW IF EXISTS bupa_gold.vw_star_members")
spark.sql("""
CREATE VIEW bupa_gold.vw_star_members AS
SELECT * FROM bupa_gold.star_members
""")

print("✅ Created view: bupa_gold.vw_star_members")

spark.table(f"{DB_GOLD}.star_members").show(10, truncate=False)


Writing star_members to: abfss://golddata@clientdatastorage.dfs.core.windows.net/star_members


✅ Wrote Delta files for star_members
✅ Registered table: bupa_gold.star_members
✅ Created view: bupa_gold.vw_star_members


+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Chronic_Disease|Employment_Status|Region_Code|dq_age_valid|dq_bmi_valid|
+------------+----------+---+------+------------------+---------------+-----------------+-----------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Asthma         |Student          |280        |1           |1           |
|MEM_00001714|P_00001714|71 |M     |27.963719840043765|None           |Employed         |360        |1           |1           |
|MEM_00001746|P_00001746|75 |M     |30.541100957674118|Diabetes       |Student          |80         |1           |1           |
|MEM_00001967|P_00001967|37 |F     |25.982850362358867|Hypertension   |Employed         |280        |1           |1           |
|MEM_00001995|P_00001995|66 |M     |32.6294573451364  |Asthma         |Employed         |360        |1  

# Cell 4 — Quick A/B/C/D style analysis on star_members

Totally optional, but matches the style you’ve used for policies/claims.

In [12]:
# Cell 4A — Snapshot metrics

sm = spark.table("bupa_gold.star_members")

metrics = {
    "rows_star_members": sm.count(),
    "distinct_member_ids": sm.select("Member_ID").distinct().count(),
    "distinct_policy_ids": sm.select("Policy_ID").distinct().count()
}

print("A) Snapshot counts:", metrics)

# --------------------------------------------------

# Cell 4B — Segment distribution (if segment available)
if "Member_Segment_Name" in sm.columns:
    print("\nB) Member segment distribution:")
    (
        sm.groupBy("Member_Segment_Name")
          .count()
          .orderBy(F.col("count").desc())
          .show(20, truncate=False)
    )
else:
    print("\nB) Member segment columns not present — skipping segment analysis.")

# --------------------------------------------------

# Cell 4C — Region vs smoker split (if Region_Name exists)
if "Region_Name" in sm.columns and "Smoker" in sm.columns:
    print("\nC) Region vs Smoker split:")
    (
        sm.groupBy("Region_Name", "Smoker")
          .count()
          .orderBy("Region_Name", "Smoker")
          .show(50, truncate=False)
    )
else:
    print("\nC) Region/Smoker columns not fully available — skipping.")

# --------------------------------------------------

# Cell 4D — Data quality flags (if present)
dq_cols = [c for c in ["dq_age_valid", "dq_bmi_valid", "dq_region_valid"] if c in sm.columns]

if dq_cols:
    print("\nD) Data quality flag coverage:")
    for c in dq_cols:
        (
            sm.groupBy(F.col(c))
              .count()
              .withColumn("pct", F.round(F.col("count") / metrics["rows_star_members"] * 100, 2))
              .show(truncate=False)
        )
else:
    print("\nD) No dq_* columns on star_members — skipping DQ breakdown.")


A) Snapshot counts: {'rows_star_members': 381109, 'distinct_member_ids': 381109, 'distinct_policy_ids': 381109}

B) Member segment columns not present — skipping segment analysis.

C) Region/Smoker columns not fully available — skipping.

D) Data quality flag coverage:
+------------+------+-----+
|dq_age_valid|count |pct  |
+------------+------+-----+
|1           |381109|100.0|
+------------+------+-----+

+------------+------+-----+
|dq_bmi_valid|count |pct  |
+------------+------+-----+
|1           |381109|100.0|
+------------+------+-----+

